In [ ]:
import json
import pandas as pd

In [ ]:
with open("/mnt/data/upcast/data/arxpr_simplified.json", "r") as f:
    json_file = json.load(f)
print(len(json_file))

In [ ]:
for key in json_file:
    fields = list(json_file[key].keys())
    break
fields


In [ ]:
dfs = {}
for field in fields:
    df = []

    for key in json_file:
        for element in json_file[key][field]:
            df.append({
                "id": key,
                "value" : element["value"],
                "ont" : None if element["ontology"] is None else element["ontology"][0],
                "ont_term" : None if element["ontology"] is None else element["ontology"][1],
                })
    df = pd.DataFrame(df)
    if not df.nunique()["ont_term"]:
        df = df[["id", "value"]]
    dfs[field] = df

In [ ]:
from collections import Counter
from matplotlib import pyplot as plt

In [ ]:
for field, n in [
    ("experimental_factors_20", 10),
    ("study_type_18", 25),
    ("type_21", 50),
]:
    commons = Counter(dfs[field]["value"]).most_common(n)
    print([c[0] for c in commons])

In [ ]:

for field in dfs:
    df = dfs[field]
    commons = Counter(df["value"]).most_common(20)
    commons = {t[0]:t[1] for t in commons}
    commons["rest"] = len(df)-sum(commons.values())
    plt.bar(commons.keys(), commons.values())
    plt.xticks(rotation=90)
    plt.title(label=field)

    plt.show()

    commons = Counter(df["value"]).most_common()
    f = len(commons)//30+1

    print("Field name:", field)
    print("Select values:", commons[::f])
    print("Number of values:", len(df))
    print(df.rename(columns={
            "id":"number of datasets with this field:",
            "value":"number of unique values:",
            "ont": "number of ontologies refered:",
            "ont_term":"number of unique ontology terms:"
            }).nunique())
    print("###"*50)


# print values for literals

In [ ]:
print("values = dict(")
for field in [
    #'sex_2',
    'hardware_4',
    'organism_part_5',
    'experimental_designs_10',
    'assay_by_molecule_14',
    'study_type_18',
    #'experimental_factors_20',
    #'type_21',
    #'adjusted_type_24'
    ]:
    df = dfs[field]
    commons = Counter(df["value"]).most_common()
    commons = [c[0] for c in commons]
    print(f"    {field} = dict(")
    print(f"        _25=", commons[:25],",")
    print(f"        _50=", commons[25:50],",")
    print(f"        _100=", commons[50:100],",")
    print(f"        _200=", commons[100:200],",")
    #print(f"        _400=", commons[200:400],",")
    print("    ),")
print(")")



# another distribution visualizing method
number of labels as fnc of number of different labels

In [ ]:

for field in [
    #'sex_2',
    'hardware_4',
    'organism_part_5',
    'experimental_designs_10',
    'assay_by_molecule_14',
    'study_type_18',
    #'experimental_factors_20',
    #'type_21',
    #'adjusted_type_24'
    ]:

    df = dfs[field]

    commons = Counter(df["value"]).most_common()

    x = []
    y = []
    accumulative_count = 0
    i = 1
    for name, number in commons:
        accumulative_count += number
        x.append(i)
        y.append(accumulative_count)
        i += 1
    
    plt.plot(x,y)
    plt.grid()
    plt.title(label=field+f", first 25: {y[min(len(y)-1, 24)]}"+f", first 50: {y[min(len(y)-1, 49)]}")
    #plt.xlim(0,250)
    plt.show()

    df = dfs[field]
    commons = Counter(df["value"]).most_common(25)
    commons = {t[0]:t[1] for t in commons}
    commons["rest"] = len(df)-sum(commons.values())
    plt.bar(commons.keys(), commons.values())
    plt.xticks(rotation=90)
    plt.title(label=field)

    plt.show()

    commons = Counter(df["value"]).most_common()
    f = len(commons)//30+1

    print("Field name:", field)
    print("Select values:", commons[::f])
    print("last values:", commons[min(len(commons)-1,24)])
    print("Number of values:", len(df))
    print(df.rename(columns={
            "id":"number of datasets with this field:",
            "value":"number of unique values:",
            "ont": "number of ontologies refered:",
            "ont_term":"number of unique ontology terms:"
            }).nunique())
    print("###"*50)



# ontology fields

In [ ]:

for field in dfs:
    df = dfs[field]


    if len(df.columns) > 2:
        print("Field name:", field)
        print("Number of values:", len(df))
        print(df.dropna(subset="ont_term").rename(columns={
                "id":"number of datasets with ontology term:",
                "value":"number of unique values with ontology term:",
                "ont": "number of ontologies refered:",
                "ont_term":"number of unique ontology terms:"
                }).nunique())
        print(df.dropna(subset="ont_term").drop("id", axis=1).drop_duplicates(subset = ["value", "ont_term"]))


## not one-to-one relationship between value and ontology term

In [ ]:
uniques_21 = dfs["type_21"].dropna(subset="ont_term").drop_duplicates(subset = ["value", "ont_term"])
uniques_21

In [ ]:
# diferent ont_terms for same value
uniques_21[uniques_21["value"] == "growth protocol"]

In [ ]:
# different value for same ont_term
uniques_21[uniques_21["ont_term"] == "efo_0003789"]

In [ ]:
# ont_term specified with eiter ":" or "_"
pd.concat([
    uniques_21[uniques_21["ont_term"] == "efo_0003789"],
    uniques_21[uniques_21["ont_term"] == "efo:0003789"]
])
    


## plot distribution, like before, but only datasets with ontology terms

In [ ]:

for field in dfs:
    if len(dfs[field].columns) > 2:

        df = dfs[field].dropna(subset="ont_term")
        commons = Counter(df["value"]).most_common(60)
        commons = {t[0]:t[1] for t in commons}
        #commons["rest"] = len(df)-sum(commons.values())
        plt.bar(commons.keys(), commons.values())
        plt.xticks(rotation=90)

        plt.show()

        commons = Counter(df["value"]).most_common()
        f = len(commons)//30+1

        print("Field name:", field)
        print("ALL values", commons)
        print("Number of values:", len(df))
        print(df.rename(columns={
                    "id":"number of datasets with ontology term:",
                    "value":"number of unique values with ontology term:",
                    "ont": "number of ontologies refered:",
                    "ont_term":"number of unique ontology terms:"
                }).nunique())


In [ ]:
# print ont terms for use in the subtre_investigtion notebook

dfs["study_type_18"].dropna(subset="ont_term").drop_duplicates(subset = ["value", "ont_term"])["ont_term"].values


## final dataset

In [ ]:
with open("/mnt/data/upcast/data/arxpr2_25_metadataset_restricted_values.json", "r") as f:
    json_file = json.load(f)
print(len(json_file))

In [ ]:
json_file

# count occurences in holdout set.

In [ ]:

import json
import pandas as pd
import matplotlib.pyplot as plt

# Load the JSON data
with open("/mnt/data/upcast/data/arxpr2_25_metadataset_train.json", 'r') as f:
    data = json.load(f)

# Detect fields from any one entry
sample_key = next(iter(data))
fields = data[sample_key].keys()

# Collect counts for each field
field_value_counts = {field: {} for field in fields}

for entry in data.values():
    for field in fields:
        values = entry[field]
        if values:  # non-empty list
            val = values[0]
            field_value_counts[field][val] = field_value_counts[field].get(val, 0) + 1

# Plot with annotations
for field, counts in field_value_counts.items():
    if counts:
        df = pd.Series(counts).sort_values(ascending=False)

        plt.figure(figsize=(10, 5))
        ax = df.plot(kind='bar')
        plt.title(f'Value Counts for Field: {field}')
        plt.xlabel('Value')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')

        # Add annotations on bars
        for i, value in enumerate(df.values):
            ax.text(i, value + 0.5, str(value), ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.show()

        print("length", len(df.values))

    break # only hardware



In [ ]:
field_names = list(data[sample_key].keys())
field_names

In [ ]:

import json
import pandas as pd
import matplotlib.pyplot as plt

# Load the JSON data
with open("/mnt/data/upcast/data/arxpr2_25_metadataset_holdout.json", 'r') as f:
    data = json.load(f)

# Detect fields from any one entry
sample_key = next(iter(data))
fields = data[sample_key].keys()

# Collect counts for each field
field_value_counts = {field: {} for field in fields}

for entry in data.values():
    for field in fields:
        values = entry[field]
        if values:  # non-empty list
            val = values[0]
            field_value_counts[field][val] = field_value_counts[field].get(val, 0) + 1

# Plot with annotations
for field, counts in field_value_counts.items():
    if counts:
        df = pd.Series(counts).sort_values(ascending=False)

        plt.figure(figsize=(10, 5))
        ax = df.plot(kind='bar')
        plt.title(f'Value Counts for Field: {field}')
        plt.xlabel('Value')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')

        # Add annotations on bars
        for i, value in enumerate(df.values):
            ax.text(i, value + 0.5, str(value), ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.show()

    break # only hardware



In [ ]:

import json
import pandas as pd
import matplotlib.pyplot as plt

# Load the JSON data
with open("/mnt/data/upcast/data/arxpr2_25_metadataset_restricted_values.json", 'r') as f:
    data = json.load(f)

# Detect fields from any one entry
sample_key = next(iter(data))
fields = data[sample_key].keys()

# Collect counts for each field
field_value_counts = {field: {} for field in fields}

for entry in data.values():
    for field in fields:
        values = entry[field]
        if values:  # non-empty list
            val = values[0]
            field_value_counts[field][val] = field_value_counts[field].get(val, 0) + 1

# Plot with annotations
for field, counts in field_value_counts.items():
    if counts:
        df = pd.Series(counts).sort_values(ascending=False)

        plt.figure(figsize=(10, 5))
        ax = df.plot(kind='bar')
        plt.title(f'Value Counts for Field: {field}')
        plt.xlabel('Value')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')

        # Add annotations on bars
        for i, value in enumerate(df.values):
            ax.text(i, value + 0.5, str(value), ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.show()

    break # only hardware



In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# Load raw JSON data
with open("/mnt/data/upcast/data/arxpr_simplified.json", "r") as f:
    raw_data = json.load(f)

# Step 1: Normalize the format
cleaned_data = {}
for key, fields in raw_data.items():
    new_fields = {}
    all_empty = True

    for field, items in fields.items():
        if field in list(data[sample_key].keys()): # only consider the 5 fields
            values = [d["value"] for d in items if "value" in d]
            if len(values) == 1:
                new_fields[field] = [values[0]]
                all_empty = False
            else:
                new_fields[field] = []

    if not all_empty:
        cleaned_data[key] = new_fields

# Step 2: Count occurrences
# (Assumes all remaining entries have same fields)
fields = next(iter(cleaned_data.values())).keys()
field_value_counts = {field: {} for field in fields}

for entry in cleaned_data.values():
    for field in fields:
        values = entry[field]
        if values:
            val = values[0]
            field_value_counts[field][val] = field_value_counts[field].get(val, 0) + 1


# Step 3: Plot top 25 value counts per field
for field, counts in field_value_counts.items():
    if counts:
        df = pd.Series(counts).sort_values(ascending=False).head(25)  # keep top 25

        plt.figure(figsize=(10, 5))
        ax = df.plot(kind='bar')
        plt.title(f'Top 25 Value Counts for Field: {field}')
        plt.xlabel('Value')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')

        # Annotate bars
        for i, value in enumerate(df.values):
            ax.text(i, value + 0.5, str(value), ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.show()


    break # only hardware

# Remake dataset

In [ ]:
# Any pmid used for training should be ignored.

#import json
#import os

#with open("/mnt/data/upcast/data/arxpr2_25_metadataset_train.json", 'r') as f:
#    original_train_file = json.load(f)

#ids_used_for_training = []
#for file in os.listdir("../profiler/all_results"):
#    pmid = file.split(".")[0]
#    if pmid in original_train_file:
#        ids_used_for_training.append(pmid)
#        
#print(len(ids_used_for_training), len(original_train_file))

In [ ]:
#print(ids_used_for_training)

In [ ]:
# this should not be updated (assuming we dont train more), since more ids will be added as used for testing
ids_used_for_training = ['20116252', '25621181', '17875202', '24124051', '18416601', '19127215', '25805847', '23079210', '19930683', '22155525', '22651900', '24441097', '33198783', '30587236', '29515032', '16768543', '25621826', '31113363', '25888729', '21535883', '21179004', '26525104', '24663216', '22586469', '24726045', '23634837', '19192944', '20618699', '25096407', '26260686', '27876829', '32102607', '27184366', '19825344', '26572553', '18522720', '26467528', '23213462', '20462447', '23409006', '19270708', '20668672', '19171122', '25165826', '26151762', '26393655', '25854683', '17121458', '27418512', '25279461', '19055731', '23209433', '21738713', '21853118', '19898461', '22257641', '19379696', '17010189', '18578861', '20011599', '34250515', '24957601', '26376863', '24037378', '25183238', '18778409', '23633485', '30461609', '19835625', '18523662', '27137948', '20044946', '22142239', '26799392', '24788094', '26095918', '26322139', '20307314', '32674113', '27471257', '26752050', '18638379', '24176859', '26270185', '23871666', '19206148', '24657902', '25569788', '19913588', '20459731', '18755030', '20360305', '25435910', '21368835', '19296830', '22152193', '24260202', '31562654', '23171372', '26443705', '25155950', '26404698', '29980666', '24603613', '25516281', '24871534', '25910039', '21483750', '19154596', '26925227', '24399302', '20926834', '27853279', '20071741', '34588551', '20797427', '19709424', '21942931', '27452608', '23671666', '31924127', '26362311', '24023716', '22761597', '25151616', '18754011', '24867642', '26516199', '18248097', '19383914', '32461585', '21912624', '20861222', '25793500', '29286103', '20678241', '25707795', '30783190', '25623184', '18400103', '25626709', '19111877', '24885787', '23809614', '22579476', '23273843', '20046837', '20176573', '23269663', '21762976', '25924916', '27851966', '19151725', '18179416', '21118511', '19514082', '24849812', '20546605', '22863408', '30277482', '20140209', '24209750', '19565513', '33540838', '28955039', '21048981', '23819599', '17060477', '28751501', '27310475', '25142253', '21085120', '28629321', '21347436', '21082716', '27411385', '23690939', '19763169', '16318412', '25010340', '20686607', '22025769', '24909122', '27899133', '23524344', '24204913', '21966449', '25209997', '18783599', '20946630', '19498199', '23863322', '26040990', '26353929', '25768297', '19889086', '23662038', '26735015', '31671153', '20105294', '19699293', '35133977', '21055414', '16446378', '29700338', '23208498', '20487527', '21533162', '21850275', '24385939', '26549805', '27003592', '24853112', '18828674', '23844193', '26882544', '24920004', '24146632', '21567994', '16756679', '25111616', '23409192', '26422514', '27337340', '18644144', '23356558', '24885405', '32527952', '23874669', '23241080', '20149220', '27031341', '23727006', '18669861', '23447580', '19561093', '22799740', '19333377', '22751464', '20361218', '26044828', '23340501', '17542647', '22355340', '26335434', '28438119', '25973918', '18307807', '21254160', '25532107', '19211664', '23155445', '31422912', '21829517', '31632976', '26863634', '21614001', '19335876', '20157446', '20078893', '29144233', '28428553', '19783824', '24687555', '27435189', '19689803', '20628647', '33658297', '24926665', '24813897', '27006701', '27725773', '24593767', '24814514', '25144761', '23239972', '28683469', '30692988', '30365341', '30816197', '22950756', '19779546', '25534508', '25590131', '24098335', '33602855', '23259508', '24718561', '18806796', '26742487', '31591573', '20939918', '20585623', '26484063', '26158897', '19445733', '26758872', '24346345', '24056944', '22885699', '18820686', '26070206', '27226156', '23301084', '30165812', '19075290', '26868491', '17828398', '22919075', '23071592', '35523142', '22144905', '20014381', '24260514', '18443592', '22393391', '28049185', '18694510', '23418630', '26305864', '28947768', '22383892', '18493317', '24349528', '25030463', '27436280', '21299862', '19852773', '25346182', '19707195', '28377501', '28824652', '27304509', '23284772', '18997868', '25425016', '23107834', '20053520', '27692071', '18673560', '31282064', '25329529', '20178741', '23651547', '21637711', '25329884', '22479649', '24561261', '18534026', '21915325', '22748054', '25915732', '21610111', '24722244']

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# -------- CONFIG --------
input_file = "/mnt/data/upcast/data/arxpr_simplified.json"
#output_file = "/mnt/data/upcast/data/arxpr3_holdout_not_normalized.json"
output_file = "/home/sondre/upcast/temp/arxpr3_holdout_not_normalized.json"
ignored_keys = ids_used_for_training



# -------- STEP 1: Load JSON --------
with open(input_file, 'r') as f:
    raw_data = json.load(f)

# -------- STEP 2: Clean & Normalize --------
intermediate_data = {}
value_counter = defaultdict(lambda: defaultdict(int))

n_ignored_keys=0
for key, fields in raw_data.items():
    if key.split("_")[0] in ignored_keys:
        n_ignored_keys += 1
        continue

    new_fields = {}
    all_empty = True

    for field, items in fields.items():
        if not field in ['hardware_4','organism_part_5','experimental_designs_10','assay_by_molecule_14','study_type_18']:
            continue
        values = [d["value"] for d in items if "value" in d]

        if not values:
            new_fields[field] = []
        elif all(v == values[0] for v in values):
            # All values the same → keep one
            new_fields[field] = [values[0]]
            value_counter[field][values[0]] += 1
            all_empty = False
        else:
            # Mixed values → discard
            new_fields[field] = []

    if not all_empty:
        intermediate_data[key] = new_fields

# -------- STEP 3: Identify Top 25 Values per Field --------
top_values_per_field = {}

for field, counter in value_counter.items():
    sorted_vals = sorted(counter.items(), key=lambda x: x[1], reverse=True)
    top_values_per_field[field] = set([v for v, _ in sorted_vals[:25]])

# -------- STEP 4: Filter to Top 25 Values per Field --------
final_data = {}

for key, fields in intermediate_data.items():
    new_fields = {}
    for field, values in fields.items():
        if values and values[0] in top_values_per_field[field]:
            new_fields[field] = values
    if new_fields:
        final_data[key] = new_fields

# -------- STEP 5: Save Cleaned + Filtered Dataset --------
with open(output_file, 'w') as f:
    json.dump(final_data, f, indent=2)

print(f"Cleaned dataset saved to '{output_file}'")
print("ignored pmids:", n_ignored_keys, ", len of cleaned data:", len(cleaned_data))


# -------- STEP 6: Plot per Field --------
# Recount values in final_data
field_value_counts = defaultdict(lambda: defaultdict(int))
fields = set()

for entry in final_data.values():
    for field, values in entry.items():
        val = values[0]
        field_value_counts[field][val] += 1
        fields.add(field)

for field in sorted(fields):
    counts = field_value_counts[field]
    df = pd.Series(counts).sort_values(ascending=False)

    plt.figure(figsize=(10, 5))
    ax = df.plot(kind='bar')
    plt.title(f'Top 25 Value Counts for Field: {field}')
    plt.xlabel('Value')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')

    # Annotate bars
    for i, value in enumerate(df.values):
        ax.text(i, value + 0.5, str(value), ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()


In [ ]:
import json
import random
from collections import defaultdict

def normalize_sample(input_file, output_file, max_per_field=300, seed=None):
    if seed is not None:
        random.seed(seed)

    # Load cleaned data
    with open(input_file, 'r') as f:
        data = json.load(f)

    items = list(data.items())
    random.shuffle(items)

    # field -> value -> list of (key, value)
    field_value_instances = defaultdict(lambda: defaultdict(list))

    for key, fields in items:
        for field, values in fields.items():
            if values:
                val = values[0]
                field_value_instances[field][val].append((key, val))

    # Final key→fields container
    sampled_keys = defaultdict(set)

    for field, value_instances in field_value_instances.items():
        available_counts = {v: len(insts) for v, insts in value_instances.items()}
        total_available = sum(available_counts.values())
        target_total = min(max_per_field, total_available)

        # Separate scarce and abundant values
        scarce = {v: c for v, c in available_counts.items() if c <= target_total // len(available_counts)}
        abundant = {v: c for v, c in available_counts.items() if v not in scarce}

        selected_counts = {}
        total_scarce = sum(scarce.values())
        remaining_budget = target_total - total_scarce

        # Add all scarce values as-is
        selected_counts.update(scarce)

        if abundant and remaining_budget > 0:
            # Split remaining budget across abundant values
            abundant_vals = list(abundant.keys())
            base = remaining_budget // len(abundant_vals)
            extra_slots = remaining_budget % len(abundant_vals)
            extra_receivers = random.sample(abundant_vals, extra_slots)

            for v in abundant_vals:
                take = base + (1 if v in extra_receivers else 0)
                selected_counts[v] = min(take, available_counts[v])  # just in case

        # Sample actual instances
        for val, count in selected_counts.items():
            instances = value_instances[val]
            random.shuffle(instances)
            for key, _ in instances[:count]:
                sampled_keys[key].add(field)

    # Rebuild final dataset
    final_data = {}

    for key, fields in data.items():
        if key not in sampled_keys:
            continue
        new_fields = {f: fields[f] for f in sampled_keys[key] if f in fields}
        if new_fields:
            for field in ['hardware_4','organism_part_5','experimental_designs_10','assay_by_molecule_14','study_type_18']:
                new_fields.setdefault(field, [])
            final_data[key] = new_fields

    # Save to output
    with open(output_file, 'w') as f:
        json.dump(final_data, f, indent=2)

    print(f"Balanced sampled dataset saved to '{output_file}' with {max_per_field} per field.")


In [ ]:
import json
import random
from collections import defaultdict

def normalize_sample(input_file, output_file, max_per_field=300, seed=None):
    if seed is not None:
        random.seed(seed)

    # Load cleaned dataset
    with open(input_file, 'r') as f:
        data = json.load(f)

    items = list(data.items())
    random.shuffle(items)

    # field -> value -> list of (key, value)
    field_value_instances = defaultdict(lambda: defaultdict(list))

    for key, fields in items:
        for field, values in fields.items():
            if values:
                val = values[0]
                field_value_instances[field][val].append((key, val))

    # Track selected keys
    sampled_keys = defaultdict(set)

    for field, value_instances in field_value_instances.items():
        # Count available per value
        available_counts = {v: len(insts) for v, insts in value_instances.items()}
        total_available = sum(available_counts.values())
        target_total = min(max_per_field, total_available)
        if target_total < max_per_field:
            print(f"Field '{field}' only had {target_total} entries available (requested {max_per_field})")

        selected_counts = defaultdict(int)
        remaining_budget = target_total

        # Step 1: Fill each value up to a baseline evenly
        value_list = list(available_counts.keys())
        random.shuffle(value_list)
        while remaining_budget > 0:
            made_progress = False
            for v in value_list:
                if selected_counts[v] < available_counts[v] and remaining_budget > 0:
                    selected_counts[v] += 1
                    remaining_budget -= 1
                    made_progress = True
            if not made_progress:
                break  # Can't allocate further

        # Step 2: Sample exact number of items
        for val, count in selected_counts.items():
            instances = value_instances[val]
            random.shuffle(instances)
            for key, _ in instances[:count]:
                sampled_keys[key].add(field)

    # Step 3: Build final dataset
    final_data = {}
    for key, fields in data.items():
        if key not in sampled_keys:
            continue
        new_fields = {f: fields[f] for f in sampled_keys[key] if f in fields}
        if new_fields:
            for field in ['hardware_4','organism_part_5','experimental_designs_10','assay_by_molecule_14','study_type_18']:
                new_fields.setdefault(field, [])

            final_data[key] = new_fields

    # Save the result
    with open(output_file, 'w') as f:
        json.dump(final_data, f, indent=2)

    print(f"Dataset saved to '{output_file}' with exactly {max_per_field} per field (when possible).")


In [ ]:
normalize_sample(
    input_file=output_file, # output from earlier
    output_file= "/home/sondre/upcast/temp/arxpr3_holdout_normalized_150.json",
    max_per_field=150
)
normalize_sample(
    input_file=output_file, # output from earlier
    output_file= "/home/sondre/upcast/temp/arxpr3_holdout_normalized_300.json",
    max_per_field=300
)

# Verify count (visualise)

In [ ]:

import json
import pandas as pd
import matplotlib.pyplot as plt

# Load the JSON data
#with open("/mnt/data/upcast/data/arxpr3_holdout_normalized_150.json", 'r') as f:
with open("/home/sondre/upcast/temp/arxpr3_holdout_normalized_150.json", 'r') as f:
    data = json.load(f)

# Detect fields from any one entry
sample_key = next(iter(data))
fields = data[sample_key].keys()

# Collect counts for each field
field_value_counts = {field: {} for field in fields}

for entry in data.values():
    for field in fields:
        values = entry[field]
        if values:  # non-empty list
            val = values[0]
            field_value_counts[field][val] = field_value_counts[field].get(val, 0) + 1

# Plot with annotations
for field, counts in field_value_counts.items():
    if counts:
        df = pd.Series(counts).sort_values(ascending=False)

        plt.figure(figsize=(10, 5))
        ax = df.plot(kind='bar')
        plt.title(f'Value Counts for Field: {field}')
        plt.xlabel('Value')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')

        # Add annotations on bars
        for i, value in enumerate(df.values):
            ax.text(i, value + 0.5, str(value), ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.show()

        print("length", len(df.values))
        print("sum:", df.sum())

    #break # only hardware



In [ ]:

import json
import pandas as pd
import matplotlib.pyplot as plt

# Load the JSON data
#with open("/mnt/data/upcast/data/arxpr3_holdout_normalized_300.json", 'r') as f:
with open("/home/sondre/upcast/temp/arxpr3_holdout_normalized_300.json", 'r') as f:
    data = json.load(f)

# Detect fields from any one entry
sample_key = next(iter(data))
fields = data[sample_key].keys()

# Collect counts for each field
field_value_counts = {field: {} for field in fields}

for entry in data.values():
    for field in fields:
        values = entry[field]
        if values:  # non-empty list
            val = values[0]
            field_value_counts[field][val] = field_value_counts[field].get(val, 0) + 1

# Plot with annotations
for field, counts in field_value_counts.items():
    if counts:
        df = pd.Series(counts).sort_values(ascending=False)

        plt.figure(figsize=(10, 5))
        ax = df.plot(kind='bar')
        plt.title(f'Value Counts for Field: {field}')
        plt.xlabel('Value')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')

        # Add annotations on bars
        for i, value in enumerate(df.values):
            ax.text(i, value + 0.5, str(value), ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.show()
        
        print("length", len(df.values))
        print("sum:", df.sum())



In [ ]:
list(data.items())